# Playground · Árboles de Decisión

**Tópicos de Inteligencia de Negocios** · Clasificación supervisada interpretable

> Los árboles hacen **preguntas encadenadas** sobre las features para llegar a una decisión. Su gran ventaja: **se pueden visualizar y explicar**. Acá podrás ver el árbol completo y cómo cambia la frontera al podarlo.

¿Qué vas a poder hacer aquí?

1. Generar datasets 2D y ver tanto la **frontera de decisión** (rectángulos) como **el árbol gráfico**.
2. Controlar la **profundidad máxima**, **criterio de división** (Gini vs Entropía), y otros parámetros de poda.
3. Ver la **importancia de cada feature** según el árbol.
4. Subir tu propio CSV.

> 🔍 **Tip:** los botones **`?`** te explican cada parámetro. Configura → presiona **🚀 Entrenar modelo**.


## Marco Teórico

### ¿Cómo se construye?
Recursivamente, en cada nodo:
1. Para cada feature, encuentra el corte que **mejor separa** las clases.
2. Mide qué tan "puro" queda cada lado con un criterio de impureza.
3. Toma el mejor corte global y crea dos ramas.
4. Repite hasta cumplir un criterio de paro (profundidad máxima, mínimo de muestras, todos los nodos puros, etc.).

### Criterios de impureza

| Criterio | Fórmula | Notas |
|----------|---------|-------|
| **Gini** | $1 - \sum p_i^2$ | Más rápido de calcular (default sklearn) |
| **Entropía** | $-\sum p_i \log_2 p_i$ | Penaliza más la mezcla — fronteras a veces ligeramente distintas |
| **Log-loss** | similar a entropía | |

> En la práctica, Gini vs Entropía dan resultados muy parecidos. Es más impactante variar `max_depth` que el criterio.

### Sobreajuste y poda
Un árbol sin restricciones **memoriza el training set**: crea una hoja para cada punto. Para prevenirlo:

- **`max_depth`** — limita la profundidad
- **`min_samples_split`** — mínimo de muestras para considerar dividir un nodo
- **`min_samples_leaf`** — mínimo de muestras en cada hoja
- **`ccp_alpha`** — post-pruning con complejidad de costo

> 💡 **Cortes axis-aligned:** los árboles dividen el espacio en **rectángulos** paralelos a los ejes. Por eso fallan con fronteras diagonales o curvas que un modelo lineal sí captura.


## 1. Configuración inicial

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_moons, make_blobs, make_circles
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.dpi'] = 90
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Listo. Librerías cargadas.')


## 2. Funciones auxiliares

In [ ]:
def con_ayuda(control, explicacion):
    btn = widgets.Button(description='?', button_style='info', tooltip=explicacion,
                         layout=widgets.Layout(width='30px', height='28px', margin='0 0 0 4px'))
    panel = widgets.HTML(
        value=(f'<div style="background:#dbeafe; padding:8px 10px; border-radius:4px; '
               f'border-left:3px solid #2563eb; margin:2px 0 8px 18px; font-size:12px; '
               f'color:#1e3a8a;">💡 {explicacion}</div>'),
        layout=widgets.Layout(display='none'))
    btn.on_click(lambda _: setattr(panel.layout, 'display',
                                   'none' if panel.layout.display != 'none' else 'block'))
    return widgets.VBox([widgets.HBox([control, btn]), panel])


def generar_dataset(tipo='Lunas', n=200, ruido=0.2, n_clases=2, seed=42):
    if tipo == 'Lunas':
        X, y = make_moons(n_samples=n, noise=ruido, random_state=seed)
    elif tipo == 'Círculos':
        X, y = make_circles(n_samples=n, noise=ruido, factor=0.4, random_state=seed)
    elif tipo == 'Blobs':
        X, y = make_blobs(n_samples=n, centers=n_clases,
                          cluster_std=ruido*4 + 0.5, random_state=seed)
    elif tipo == 'Classification':
        X, y = make_classification(n_samples=n, n_features=2, n_redundant=0,
                                   n_informative=2, n_clusters_per_class=1,
                                   n_classes=n_clases, flip_y=ruido*0.3,
                                   class_sep=2.0 - ruido, random_state=seed)
    return X, y


def construir_arbol(max_depth, criterion, min_samples_split,
                    min_samples_leaf, max_features, ccp_alpha):
    return DecisionTreeClassifier(
        max_depth=None if max_depth == 0 else max_depth,
        criterion=criterion,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=None if max_features == 'todas' else max_features,
        ccp_alpha=ccp_alpha,
        random_state=42,
    )


def graficar_resultados_arbol(X, y, modelo, X_train, X_test, y_train, y_test,
                               feat_names=('Feature 1', 'Feature 2'), titulo=''):
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3,
                          height_ratios=[1, 1, 0.8])
    ax1 = fig.add_subplot(gs[0, :2])
    ax2 = fig.add_subplot(gs[0, 2])
    ax3 = fig.add_subplot(gs[1:, :])

    # ---- Frontera de decisión (rectangular) ----
    margen = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:,0].min()-margen, X[:,0].max()+margen, 200),
        np.linspace(X[:,1].min()-margen, X[:,1].max()+margen, 200))
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    n_clases = len(np.unique(y))
    cmap_back = plt.cm.RdYlBu_r if n_clases == 2 else plt.cm.Set3
    cmap_pts = plt.cm.RdYlBu_r if n_clases == 2 else plt.cm.Set1
    ax1.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_back)
    ax1.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap=cmap_pts, s=40,
                edgecolor='white', linewidth=0.5, alpha=0.85, label='Train')
    ax1.scatter(X_test[:,0], X_test[:,1], c=y_test, cmap=cmap_pts, s=80,
                marker='s', edgecolor='black', linewidth=1.0, alpha=0.95, label='Test')
    ax1.set_title(f'Frontera de decisión {titulo}', fontsize=12, fontweight='bold')
    ax1.set_xlabel(feat_names[0]); ax1.set_ylabel(feat_names[1])
    ax1.legend(loc='best')

    # ---- Métricas + confusión + importancias ----
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    avg = 'binary' if n_clases == 2 else 'macro'
    ax2.imshow(cm, cmap='Blues', aspect='auto')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax2.text(j, i, cm[i,j], ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black',
                     fontsize=12, fontweight='bold')
    ax2.set_title('Matriz de Confusión', fontsize=11, fontweight='bold')
    ax2.set_xticks(range(cm.shape[1])); ax2.set_yticks(range(cm.shape[0]))
    ax2.set_xticklabels([f'P{i}' for i in range(cm.shape[1])])
    ax2.set_yticklabels([f'R{i}' for i in range(cm.shape[0])])
    ax2.grid(False)

    # ---- Árbol gráfico ----
    ax3.set_title(f'Árbol completo — profundidad real: {modelo.get_depth()}, '
                  f'hojas: {modelo.get_n_leaves()}', fontsize=11, fontweight='bold')
    plot_tree(modelo, ax=ax3, feature_names=list(feat_names),
              class_names=[f'Clase {i}' for i in range(n_clases)],
              filled=True, rounded=True, fontsize=9,
              proportion=False, impurity=True)

    # Mostrar métricas y feature importance como texto
    metricas = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precisión': precision_score(y_test, y_pred, average=avg, zero_division=0),
        'Recall': recall_score(y_test, y_pred, average=avg, zero_division=0),
        'F1': f1_score(y_test, y_pred, average=avg, zero_division=0),
    }
    plt.tight_layout(); plt.show()
    print('📊 Métricas (Test):')
    for k, v in metricas.items():
        print(f'  {k:<12s}: {v:.4f}')
    print('\n🎯 Importancia de features:')
    for fn, imp in zip(feat_names, modelo.feature_importances_):
        bar = '█' * int(imp * 30)
        print(f'  {fn:<20s} {imp:.4f}  {bar}')


## 3. Playground · Datos sintéticos 🧪

> 🔍 Botones **`?`** para ayuda. Configura → **🚀 Entrenar modelo**.


In [ ]:
# DATOS
w_tipo = widgets.Dropdown(options=['Lunas','Círculos','Blobs','Classification'],
                          value='Lunas', description='Dataset:',
                          style={'description_width':'130px'})
w_n = widgets.IntSlider(value=200, min=50, max=600, step=10,
                        description='n puntos:', style={'description_width':'130px'})
w_ruido = widgets.FloatSlider(value=0.2, min=0.0, max=0.6, step=0.05,
                              description='Ruido:', style={'description_width':'130px'})
w_clases = widgets.IntSlider(value=2, min=2, max=4,
                             description='Clases:', style={'description_width':'130px'})
w_seed = widgets.IntSlider(value=42, min=0, max=100,
                           description='Semilla:', style={'description_width':'130px'})

# MODELO
w_depth = widgets.IntSlider(value=3, min=0, max=15, step=1,
                            description='max_depth:', style={'description_width':'130px'})
w_crit = widgets.Dropdown(options=['gini','entropy','log_loss'], value='gini',
                          description='criterion:', style={'description_width':'130px'})
w_split_min = widgets.IntSlider(value=2, min=2, max=50,
                                description='min_samples_split:',
                                style={'description_width':'130px'})
w_leaf_min = widgets.IntSlider(value=1, min=1, max=30,
                               description='min_samples_leaf:',
                               style={'description_width':'130px'})
w_feat = widgets.Dropdown(options=['todas','sqrt','log2'], value='todas',
                          description='max_features:', style={'description_width':'130px'})
w_ccp = widgets.FloatSlider(value=0.0, min=0.0, max=0.1, step=0.005,
                            description='ccp_alpha:', style={'description_width':'130px'})
w_split = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                              description='Test size:', style={'description_width':'130px'})

# Explicaciones
ay_tipo = 'Forma del dataset. Lunas y Círculos ponen a prueba los cortes axis-aligned del árbol.'
ay_n = 'Cantidad de puntos. Pocos puntos + árbol profundo = sobreajuste casi garantizado.'
ay_ruido = 'Más ruido = clases más mezcladas = el árbol crece más para "memorizar".'
ay_clases = 'Número de clases (solo Blobs y Classification).'
ay_seed = 'Misma semilla = mismos datos.'
ay_depth = ('Profundidad MÁXIMA del árbol. 0 = sin límite (sobreajuste casi seguro). '
            '3-5 suele dar buenos resultados. Sube y mira cómo crece el árbol.')
ay_crit = ('Cómo mide el árbol qué tan "mezclada" está cada hoja. Gini y Entropy '
           'suelen dar resultados muy similares en la práctica.')
ay_split_min = ('Mínimo de muestras para considerar dividir un nodo. Súbelo para forzar '
                'al árbol a hacer divisiones solo cuando hay suficiente evidencia.')
ay_leaf_min = ('Mínimo de muestras en CADA hoja. Súbelo para que no haya hojas "lobuga" '
               'con 1-2 muestras (que casi siempre son ruido).')
ay_feat = ('Cuántas features se consideran al buscar la mejor división. "todas" es '
           'el default; "sqrt"/"log2" se usan en Random Forests para diversidad.')
ay_ccp = ('Post-pruning con costo de complejidad. Subir ccp_alpha poda más ramas. '
          'Es una forma elegante de simplificar el árbol después de entrenarlo.')
ay_split = '0.25 = 75% train / 25% test.'

panel_d = widgets.VBox([
    widgets.HTML('<b>📊 Datos</b>'),
    con_ayuda(w_tipo, ay_tipo), con_ayuda(w_n, ay_n),
    con_ayuda(w_ruido, ay_ruido), con_ayuda(w_clases, ay_clases),
    con_ayuda(w_seed, ay_seed),
])
panel_m = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_depth, ay_depth), con_ayuda(w_crit, ay_crit),
    con_ayuda(w_split_min, ay_split_min), con_ayuda(w_leaf_min, ay_leaf_min),
    con_ayuda(w_feat, ay_feat), con_ayuda(w_ccp, ay_ccp),
    con_ayuda(w_split, ay_split),
])
controles = widgets.HBox([panel_d, panel_m])
salida = widgets.Output()

def actualizar():
    with salida:
        clear_output(wait=True)
        try:
            X, y = generar_dataset(w_tipo.value, w_n.value, w_ruido.value,
                                    w_clases.value, w_seed.value)
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_split.value,
                                                       random_state=42, stratify=y)
            arbol = construir_arbol(w_depth.value, w_crit.value, w_split_min.value,
                                     w_leaf_min.value, w_feat.value, w_ccp.value)
            arbol.fit(X_tr, y_tr)
            graficar_resultados_arbol(
                X, y, arbol, X_tr, X_te, y_tr, y_te,
                titulo=f'· depth={w_depth.value if w_depth.value>0 else "∞"}, '
                       f'{w_crit.value}, ccp={w_ccp.value:.3f}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                       layout=widgets.Layout(width='220px', height='40px',
                                             margin='10px 0 6px 0'))
estado = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                            'Configura los parámetros y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale(*_):
    estado.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b> '
                    'Haz clic en <b>Entrenar modelo</b>.</span>')

def _go(_):
    btn_e.disabled = True; btn_e.description = '⏳ Entrenando...'
    estado.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar()
        estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b> '
                        'Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e.disabled = False; btn_e.description = '🚀 Entrenar modelo'

btn_e.on_click(_go)
for w in [w_tipo, w_n, w_ruido, w_clases, w_seed, w_depth, w_crit,
          w_split_min, w_leaf_min, w_feat, w_ccp, w_split]:
    w.observe(_stale, names='value')

display(controles, btn_e, estado, salida)
actualizar()
estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado con la configuración inicial.</b></span>')


## 4. Playground · Sube tu CSV 📂

Sube un CSV con 2 columnas numéricas (features) y 1 con etiqueta categórica.


In [ ]:
estado_csv = {'df': None}

w_upload = widgets.FileUpload(accept='.csv', multiple=False, description='📁 Subir CSV')
w_x1 = widgets.Dropdown(options=[], description='Feature 1:', style={'description_width':'130px'})
w_x2 = widgets.Dropdown(options=[], description='Feature 2:', style={'description_width':'130px'})
w_yc = widgets.Dropdown(options=[], description='Etiqueta:', style={'description_width':'130px'})

w_depth2 = widgets.IntSlider(value=3, min=0, max=15,
                             description='max_depth:', style={'description_width':'130px'})
w_crit2 = widgets.Dropdown(options=['gini','entropy','log_loss'], value='gini',
                           description='criterion:', style={'description_width':'130px'})
w_leaf2 = widgets.IntSlider(value=1, min=1, max=30,
                            description='min_samples_leaf:',
                            style={'description_width':'130px'})
w_ccp2 = widgets.FloatSlider(value=0.0, min=0.0, max=0.1, step=0.005,
                             description='ccp_alpha:', style={'description_width':'130px'})
w_s2 = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                           description='Test size:', style={'description_width':'130px'})

salida_csv = widgets.Output()
salida_info = widgets.Output()

def on_upload(change):
    with salida_info:
        clear_output(wait=True)
        if not w_upload.value: return
        try:
            archivo = w_upload.value[0] if isinstance(w_upload.value, tuple) else next(iter(w_upload.value.values()))
            contenido = archivo['content']
            nombre = archivo.get('name','archivo.csv')
        except Exception:
            archivo = list(w_upload.value.values())[0]
            contenido = archivo['content']
            nombre = archivo.get('metadata',{}).get('name','archivo.csv')
        try:
            df = pd.read_csv(io.BytesIO(bytes(contenido)))
        except Exception as e:
            print(f'⚠️ {e}'); return
        estado_csv['df'] = df
        cn = df.select_dtypes(include=[np.number]).columns.tolist()
        cc = df.columns.tolist()
        if len(cn) < 2:
            print(f'⚠️ Necesitas ≥ 2 columnas numéricas. Encontré: {cn}'); return
        w_x1.options = cn; w_x2.options = cn; w_yc.options = cc
        w_x1.value = cn[0]; w_x2.value = cn[1]; w_yc.value = cc[-1]
        print(f'✅ {nombre} — {df.shape[0]}×{df.shape[1]}')
        display(df.head())

w_upload.observe(on_upload, names='value')

def actualizar2():
    with salida_csv:
        clear_output(wait=True)
        df = estado_csv['df']
        if df is None: print('⬆️ Sube un CSV primero.'); return
        try:
            sub = df[[w_x1.value, w_x2.value, w_yc.value]].dropna()
            X = sub[[w_x1.value, w_x2.value]].values
            y_raw = sub[w_yc.value].values
            unicos, y = np.unique(y_raw, return_inverse=True)
            if len(unicos) < 2:
                print('⚠️ La etiqueta debe tener ≥ 2 valores.'); return
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_s2.value,
                                                       random_state=42, stratify=y)
            arbol = construir_arbol(w_depth2.value, w_crit2.value, 2,
                                     w_leaf2.value, 'todas', w_ccp2.value)
            arbol.fit(X_tr, y_tr)
            graficar_resultados_arbol(
                X, y, arbol, X_tr, X_te, y_tr, y_te,
                feat_names=(w_x1.value, w_x2.value),
                titulo=f'· {w_yc.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e2 = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                        layout=widgets.Layout(width='220px', height='40px',
                                              margin='10px 0 6px 0'))
estado2 = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                             'Sube un CSV y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale2(*_):
    if estado_csv['df'] is None: return
    estado2.value = '<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b></span>'

def _go2(_):
    if estado_csv['df'] is None:
        estado2.value = '<span style="color:#dc2626;">⚠️ Primero sube un CSV.</span>'; return
    btn_e2.disabled = True; btn_e2.description = '⏳ Entrenando...'
    estado2.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar2()
        estado2.value = '<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b></span>'
    except Exception as e:
        estado2.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e2.disabled = False; btn_e2.description = '🚀 Entrenar modelo'

btn_e2.on_click(_go2)
for w in [w_x1, w_x2, w_yc, w_depth2, w_crit2, w_leaf2, w_ccp2, w_s2]:
    w.observe(_stale2, names='value')

panel_c = widgets.VBox([
    widgets.HTML('<b>📁 Datos (CSV)</b>'),
    con_ayuda(w_upload, 'Sube tu .csv (2 features numéricas + 1 etiqueta).'),
    con_ayuda(w_x1, 'Primera variable predictora.'),
    con_ayuda(w_x2, 'Segunda variable predictora.'),
    con_ayuda(w_yc, 'Variable a predecir (categórica).'),
])
panel_m2 = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_depth2, ay_depth), con_ayuda(w_crit2, ay_crit),
    con_ayuda(w_leaf2, ay_leaf_min), con_ayuda(w_ccp2, ay_ccp),
    con_ayuda(w_s2, ay_split),
])
display(widgets.HBox([panel_c, panel_m2]), btn_e2, estado2, salida_info, salida_csv)


## 5. Ejercicios guiados 📝

### Ejercicio 1 — El árbol que memoriza
1. Dataset = **Lunas**, n = 200, ruido = 0.3.
2. Entrena con `max_depth = 0` (sin límite). Mira el árbol gigante.
3. Anota la accuracy de train (la verás implícita) y de test.
4. **Pregunta:** ¿Cuántas hojas tiene? ¿Cómo se ve la frontera? ¿Es razonable?

### Ejercicio 2 — La poda al rescate
1. Mismo dataset. Reduce `max_depth` de 0 → 10 → 5 → 3 → 2.
2. **Pregunta:** ¿En qué profundidad la frontera deja de verse "ruidosa"? ¿Cómo cambia accuracy de test?

### Ejercicio 3 — Gini vs Entropía
1. Dataset = **Classification**, ruido = 0.3, max_depth = 4.
2. Compara `gini` vs `entropy`.
3. **Pregunta:** ¿Notas alguna diferencia? Es muy pequeña — un buen recordatorio de que el criterio importa menos que la profundidad.

### Ejercicio 4 — Cortes axis-aligned
1. Dataset = **Círculos**, ruido = 0.1.
2. Prueba max_depth = 3, luego 8, luego 15.
3. **Pregunta:** ¿Qué tan bien puede aproximar un árbol una frontera circular? Compara con lo que harías con regresión logística + features polinomiales.

### Ejercicio 5 — Importancia de features
1. Dataset = **Classification**, n_clases = 3, ruido = 0.2, max_depth = 4.
2. Mira la sección "🎯 Importancia de features" abajo.
3. **Pregunta:** ¿Las dos features tienen importancia similar? ¿Qué pasa cuando una es claramente más informativa?

### Ejercicio 6 — Tu propio CSV
Sube un dataset (Titanic, Iris, etc.) y construye un árbol interpretable de profundidad 3-4. Analiza las reglas que aparecen en el árbol gráfico.


## 6. Resumen

- Los árboles son **interpretables**: cada decisión es una pregunta legible.
- Sin restricciones, **sobreajustan brutalmente** — siempre poda con `max_depth` o `ccp_alpha`.
- Dividen el espacio en **rectángulos axis-aligned** — fallan con fronteras diagonales o curvas suaves.
- **Inestables**: pequeños cambios en los datos producen árboles muy distintos (por eso existen los Random Forests).
- Importancia de features es un subproducto gratuito muy útil para análisis exploratorio.

---

> *Tópicos de Inteligencia de Negocios · Playground de Machine Learning*
